In [3]:
from dotenv import load_dotenv
load_dotenv()

from groq import Groq
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda

client = Groq()
model = ChatGroq(model="openai/gpt-oss-120b")

def transcribe(path: str) -> dict:
    with open(path, "rb") as f:
        result = client.audio.transcriptions.create(
            file=f, model="whisper-large-v3", language="ar"
        )
    return {"text": result.text}

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a sentiment analysis system. Classify the user's text "
               "as exactly one word: Positive, Negative, or Neutral. Reply with that word only."),
    ("user", "{text}"),
])

full_chain = RunnableLambda(transcribe) | prompt | model | StrOutputParser()
print(full_chain.invoke("voice_note.mp3"))

Neutral
